# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shweta1805/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Warehouse connection ready.")

Warehouse connection ready.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Build the feature vector

The feature vector uses historical search and engagement signals that are available before the prediction moment. I use GSC impressions, GSC clicks, average position, GA4 sessions, and engaged sessions. Missing numeric values are filled with 0 after checking data availability, and the features are kept numeric for model use.


In [10]:
# Build the feature dataframe

feature_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM {TABLES["fact_daily"]}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
LIMIT 10000
"""

features = con.sql(feature_query).df()

numeric_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

features[numeric_features] = features[numeric_features].fillna(0)

print("Feature rows:", len(features))
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 10000


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,0,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,0,0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,0,0
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,0,0
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,0,0


In [11]:
con.sql(f"""
    SELECT COUNT(*) AS rows
    FROM {TABLES["fact_daily"]}
    WHERE month = '2026-03'
""").df()

,rows
0,9841378


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

| Feature | Meaning | Missing-value handling | Available when? |
|---|---|---|---|
| `gsc_impressions` | Number of times the content appeared in Google Search. | Missing values are filled with 0 after filtering for available GSC data. | Available from the historical observation period. |
| `gsc_clicks` | Number of Google Search clicks received by the content. | Missing values are filled with 0. | Available from the historical observation period. |
| `gsc_avg_position` | Average search position observed for the content. | Missing values are filled with 0; this should be interpreted carefully because 0 is not a real search position. | Available from the historical observation period. |
| `ga4_sessions` | Number of website sessions associated with the content. | Missing values are filled with 0. | Available when GA4 data is available for the observation. |
| `ga4_engaged_sessions` | Number of sessions showing engagement with the content. | Missing values are filled with 0. | Available when GA4 data is available for the observation. |

The identifiers `client_hash_id` and `content_hash_id` are retained for grouping and traceability but are not treated as predictive features.

In [12]:
# Check missing values in the feature dataframe

feature_check = features[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions"
    ]
].isna().sum().to_frame("missing_values")

feature_check

,missing_values
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_sessions,0
ga4_engaged_sessions,0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage check

The selected features are calculated from the historical observation period only. I exclude future-period performance and label-derived fields from the feature set.

I also check the feature dates directly so that the inputs do not extend beyond the intended observation window. Client and content identifiers are retained for grouping but are not used as predictive features.

This is a practical leakage check: a feature is acceptable only if its value would be known at the moment the prediction is made.

In [13]:
# Check the full observation window for leakage

date_check = con.sql(f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS distinct_dates
    FROM {TABLES["fact_daily"]}
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
""").df()

date_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_date,last_date,distinct_dates
0,2026-03-01,2026-03-31,31


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### What I excluded and why

- **Future-period performance:** excluded because it would reveal information from after the prediction moment and cause leakage.
- **Label-derived fields:** excluded because they are calculated from the outcome we are trying to predict.
- **Client and content identifiers as predictive features:** excluded because the model could memorize identifiers instead of learning generalizable performance patterns.
- **Report date as a predictive feature:** retained for time-based filtering and validation, but not treated as a direct predictive signal.
- **Data-availability flags as predictive features:** used for filtering and data-quality checks rather than as performance signals.

In [14]:
# Show the fields intentionally excluded from the predictive feature set

excluded_fields = [
    "future-period performance",
    "label-derived fields",
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "data-availability flags"
]

print("Excluded fields / field groups:")
for field in excluded_fields:
    print("-", field)

Excluded fields / field groups:
- future-period performance
- label-derived fields
- client_hash_id
- content_hash_id
- report_date
- data-availability flags


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.